# 09 · Data catalogue

What every raw source contains: reporter recordings (sampling, duration, value range, duplicates between sources), cell models and their CCLE ids, Circa-SCOPE conditions, drug-response files and patient cohorts.

Code: `src/data/audit.py` (`catalog()`); the same tables are written by `python run.py audit --catalog`. No model is needed; it runs in well under a minute and uses little memory (sheets are sized from the workbook directory, large matrices are opened for their header only).

Outputs: `data/catalog/` (`sources.csv`, `records.csv`, `models.csv`, `circascope_sheets.csv`, `drug_files.csv`, `cohorts.csv`, `SUMMARY.md`).

In [ ]:
import sys
from pathlib import Path
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists() and (p / "configs").exists())
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data import audit as ad
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 80)

cat = ad.catalog()
print((ROOT / "data" / "catalog" / "SUMMARY.md").read_text(encoding="utf-8"))

## 1. Sources

In [ ]:
cat["sources"]

## 2. Reporter recordings

`duplicate_of` marks a record whose values already occur in an earlier source; `raw_like` is false when a series contains non-positive values (detrended or baseline-subtracted); `usable` = not a duplicate and at least 72 h.

In [ ]:
r = cat["records"]
r.groupby("source").agg(records=("record_id", "size"), new=("duplicate_of", lambda s: (s == "").sum()),
                        usable=("usable", "sum"), raw_like=("raw_like", "mean"), models=("cell_model", "nunique"),
                        dt_h=("dt_h", "median"), duration_h=("duration_h", "median")).round(3)

## 3. Cell models

Models sharing a `lineage_group` come from the same patient or parental line and must stay together in splits and statistics.

In [ ]:
m = cat["models"]
m[["source", "cell_model", "cancer_type", "species", "lineage_group", "ccle_id", "reporters", "n_records", "n_usable",
   "raw_like", "n_dates", "median_duration_h"]]

In [ ]:
core = m[~m.genotype_variant & m.source.isin(["msb2025", "natcomm2024", "nb2026"])]
core = core.groupby("cell_model").agg(cancer_type=("cancer_type", "first"), lineage_group=("lineage_group", "first"),
                                      ccle_id=("ccle_id", "max"), n_usable=("n_usable", "sum"))
core = core[core.n_usable > 0]
print("core models:", len(core), "| lineage groups:", core.lineage_group.nunique(), "| with CCLE id:", int((core.ccle_id != "").sum()))
print("without CCLE id in ccle_broad_2019:", core[core.ccle_id == ""].index.tolist())
core.groupby("cancer_type").agg(models=("lineage_group", "size"), lineage_groups=("lineage_group", "nunique"),
                                with_ccle=("ccle_id", lambda s: (s != "").sum()), usable_records=("n_usable", "sum"))

## 4. Circa-SCOPE conditions

In [ ]:
cs = cat["circascope_sheets"]
cs.groupby(["file", "agent"]).agg(sheets=("sheet", "size"), cells=("n_cells", "sum"), doses=("dose", ", ".join))

## 5. Drug-response files

In [ ]:
d = cat["drug_files"]
d.groupby(["source", "folder"]).agg(files=("file", "size"), models=("cell_model", lambda s: s[s != ""].nunique()),
                                    drugs=("drug", lambda s: ", ".join(sorted(set(s) - {""}))))

## 6. Patient cohorts

In [ ]:
cat["cohorts"]

## 7. Harmonised tables

`data/processed/external_v1/`: one long drug-response table, the expression of the catalogued cell models (DepMap) and the model table saying which analyses each model can enter. Same as `python run_exp02.py tables`.

In [ ]:
from src.data import external as fx
dr, ex, models = fx.tables()

In [ ]:
models[["cell_model", "cancer_type", "lineage_group", "n_usable_records", "has_expression",
        "n_drugs_tod_max_range", "n_drugs_dr_GR50", "n_drugs_gdsc_auc"]]